<a href="https://colab.research.google.com/github/madalamanikanta/ImageCaptioning_MiniProject/blob/manikanta-dev/05_Feature_Verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1 — Connect Google Drive


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## Step 2 — Import Required Libraries


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

## Step 3 — Define Project Paths


In [3]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/ImageCaptioning_MiniProject"
)

PROCESSED_DIR = PROJECT_DIR / "Processed"

FEATURE_DIR = (
    PROJECT_DIR
    / "Features"
    / "CLIP"
)

print("Project directory:", PROJECT_DIR)
print("Processed directory:", PROCESSED_DIR)
print("Feature directory:", FEATURE_DIR)

Project directory: /content/drive/MyDrive/ImageCaptioning_MiniProject
Processed directory: /content/drive/MyDrive/ImageCaptioning_MiniProject/Processed
Feature directory: /content/drive/MyDrive/ImageCaptioning_MiniProject/Features/CLIP


## Step 4 — Verify Required Files and Folders


In [4]:
print("Project exists :", PROJECT_DIR.exists())
print("Processed exists :", PROCESSED_DIR.exists())
print("Feature directory exists :", FEATURE_DIR.exists())

cleaned_file = PROCESSED_DIR / "cleaned_dataset.csv"

print("Cleaned dataset exists :", cleaned_file.exists())

Project exists : True
Processed exists : True
Feature directory exists : True
Cleaned dataset exists : True


## Step 5 — Load Cleaned Dataset


In [5]:
df = pd.read_csv(cleaned_file)
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (599278, 3)


,image_path,caption,dataset
0,/content/drive/MyDrive/ImageCaptioning_MiniPro...,<start> A child in a pink dress is climbing up...,Flickr8K
1,/content/drive/MyDrive/ImageCaptioning_MiniPro...,<start> A girl going into a wooden building <end>,Flickr8K
2,/content/drive/MyDrive/ImageCaptioning_MiniPro...,<start> A little girl climbing into a wooden p...,Flickr8K
3,/content/drive/MyDrive/ImageCaptioning_MiniPro...,<start> A little girl climbing the stairs to h...,Flickr8K
4,/content/drive/MyDrive/ImageCaptioning_MiniPro...,<start> A little girl in a pink dress going in...,Flickr8K


## Step 6 — Create Unique Image List


In [6]:
unique_images = (
    df["image_path"]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Total dataset rows :", len(df))
print("Unique images      :", len(unique_images))

Total dataset rows : 599278
Unique images      : 119865


## Step 7 — Find CLIP Feature Files


In [7]:
feature_files = sorted(
    FEATURE_DIR.glob("features_*.npy")
)

print("Feature files found:", len(feature_files))

if feature_files:
    print("First file:", feature_files[0].name)
    print("Last file :", feature_files[-1].name)

Feature files found: 120
First file: features_0000.npy
Last file : features_0119.npy


## Step 8 — Verify Feature Dimensions


In [8]:
chunk_info = []

for feature_file in feature_files:

    features = np.load(feature_file)

    chunk_info.append({
        "file": feature_file.name,
        "rows": features.shape[0],
        "dimensions": features.shape[1]
    })

chunk_info_df = pd.DataFrame(chunk_info)

chunk_info_df.head()

,file,rows,dimensions
0,features_0000.npy,1000,512
1,features_0001.npy,1000,512
2,features_0002.npy,1000,512
3,features_0003.npy,1000,512
4,features_0004.npy,1000,512


## Step 9 — Verify 512-Dimensional Features


In [9]:
invalid_dimensions = chunk_info_df[
    chunk_info_df["dimensions"] != 512
]

print(
    "Chunks with incorrect dimensions:",
    len(invalid_dimensions)
)

if len(invalid_dimensions) == 0:
    print("All feature chunks have 512 dimensions.")
else:
    print(invalid_dimensions)

Chunks with incorrect dimensions: 0
All feature chunks have 512 dimensions.


## Step 10 — Verify Total Number of Feature Vectors


In [10]:
total_features = chunk_info_df["rows"].sum()

print("Total feature vectors :", total_features)
print("Total unique images   :", len(unique_images))

if total_features == len(unique_images):
    print("SUCCESS: Feature count matches unique image count.")
else:
    print("WARNING: Feature count does not match.")

Total feature vectors : 119865
Total unique images   : 119865
SUCCESS: Feature count matches unique image count.


## Step 11 — Check Feature Values


In [11]:
nan_count = 0
inf_count = 0

for feature_file in feature_files:

    features = np.load(feature_file)

    nan_count += np.isnan(features).sum()
    inf_count += np.isinf(features).sum()

print("Total NaN values :", nan_count)
print("Total Inf values :", inf_count)

Total NaN values : 0
Total Inf values : 0


## Step 12 — Verify Feature Normalization


In [12]:
# Check the first feature chunk

sample_features = np.load(
    feature_files[0]
)

norms = np.linalg.norm(
    sample_features,
    axis=1
)

print("Minimum norm :", norms.min())
print("Maximum norm :", norms.max())
print("Mean norm    :", norms.mean())

Minimum norm : 8.948049
Maximum norm : 12.135174
Mean norm    : 10.490043


## Step 13 — Inspect a Single CLIP Feature


In [13]:
sample_feature = sample_features[0]

print("Image:")
print(unique_images.iloc[0])

print()
print("Feature shape:")
print(sample_feature.shape)

print()
print("First 20 feature values:")
print(sample_feature[:20])

Image:
/content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/flickr8k/Images/1000268201_693b08cb0e.jpg

Feature shape:
(512,)

First 20 feature values:
[-1.1683404e-03  9.3036175e-02 -8.8475421e-02  2.4126199e-01
  4.1285539e-01  3.6548114e-01  4.4905567e-01 -2.2327733e-01
 -6.9190347e-01 -2.9288578e-01  2.2011417e-01  2.7862102e-02
 -2.4543113e-01 -1.8541233e-01  7.8741565e-02 -3.2384026e-01
 -1.3402278e+00 -7.0447445e-02  2.7399939e-01 -8.0469169e-02]


## Step 14 — Final Feature Verification Summary


In [14]:
print("=" * 60)
print("FINAL CLIP FEATURE VERIFICATION")
print("=" * 60)

print("Caption rows          :", len(df))
print("Unique images         :", len(unique_images))
print("Feature files         :", len(feature_files))
print("Feature vectors       :", total_features)
print("Feature dimensions    :", 512)
print("NaN values            :", nan_count)
print("Infinite values       :", inf_count)

print()

if (
    total_features == len(unique_images)
    and len(invalid_dimensions) == 0
    and nan_count == 0
    and inf_count == 0
):

    print("STATUS: CLIP FEATURES VERIFIED SUCCESSFULLY")

else:

    print("STATUS: CHECK REQUIRED")

FINAL CLIP FEATURE VERIFICATION
Caption rows          : 599278
Unique images         : 119865
Feature files         : 120
Feature vectors       : 119865
Feature dimensions    : 512
NaN values            : 0
Infinite values       : 0

STATUS: CLIP FEATURES VERIFIED SUCCESSFULLY
